In [ ]:
import time
import math
import os
from typing import Dict, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# Reproducibility 
# ------------------------------------------------------------------
GLOBAL_SEED = 42

torch.manual_seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

torch.set_num_threads(
    min(6, os.cpu_count() or 1)
)

DTYPE = torch.float64
DEVICE = torch.device("cpu")

# print("device:", DEVICE)
# print("dtype :", DTYPE)
# print("seed  :", GLOBAL_SEED)


# Classical deep 2BSDE for a Bayesian sensing/control problem

This notebook is deliberately self-contained. It explains a measure-valued stochastic optimal control problem, mostly from the perspective of **controlled stochastic filtering theory**, and the associated**classical fixed-auxiliary deep 2BSDE numerical method**.

The notebook covers:
1. hidden signal, observation, posterior, and control;
2. the sensor $h$;
3. the KS posterior dynamics;
4. the cost functional;
5. the HJB and Hamiltonian;
6. the classical fixed-auxiliary 2BSDE derivation;
7. the neural network design; 
8. implementation; 

# Main notation

| Symbol | Meaning | Dimension |
|---|---|---:|
| $$T$$ | terminal time | $$\text{scalar}{}$$ |
| $$X$$ | hidden static signal | $$d$$ |
| $$x_i$$ | possible hidden state | $$d$$ |
| $$u_t$$ | control | $$n$$ |
| $$Y_t$$ | observation state | $$m$$ |
| $$P_t$$ | posterior probability vector | $$N$$ |
| $$S_t=(Y_t,P_t)$$ | concatenated state | $$m + N$$ |
| $$h^{(i)}(u,y)=h(u,y,x_i)$$ | sensor response at state $x_i$ | $$m$$ |
| $$\bar h(y,p,u)=\sum_{i=1}^N p_i h^{(i)}(u,y)$$ | posterior mean sensor response | $$m$$ |
| $$B(y,p,u)$$ | posterior diffusion matrix | $$N \times m$$ |
| $$b(s,u)$$ | $S$ drift | $$m+N$$ |
| $$\Sigma(s,u)$$ | $S$ diffusion | $$(m+N) \times m$$ |
| $$k(s,u)$$ | running cost | $$\text{scalar}{}$$ |
| $$g(s)$$ | terminal cost | $$\text{scalar}{}$$ |
| $$J(u)$$ | expected total cost | $$\text{scalar}{}$$ |
| $$V(t,s)$$ | value function | $$\text{scalar}{}$$ |
| $$z=D_sV$$ | value gradient | $$m+N$$ |
| $$\Gamma=D_s^2V$$ | value Hessian | $$(m+N) \times (m+N)$$ |
| $$H(s,z,\Gamma,u)$$ | Hamiltonian for one control | $$\text{scalar}{}$$ |
| $$\mathcal H(s,z,\Gamma)$$ | minimised Hamiltonian | $$\text{scalar}{}$$ |
| $$u^{aux}_{}$$ | fixed auxiliary/reference control | $$n$$ |
| $$R_t=V(t,S_t^{\rm aux})$$ | scalar pathwise value | $$\text{scalar}{}$$ |
| $$Z_t=D_sV(t,S_t^{\rm aux})$$ | pathwise gradient | $$m+N$$ |
| $$A_t$$ | drift of $Z_t$ | $$m+N$$ |
| $$\Gamma_t$$ | pathwise Hessian | $$(m+N) \times (m+N)$$ |
| $$I_t$$ | innovation Brownian motion | $$m$$ |

# 1. Hidden signal, observation, control, and posterior
The unknown static signal is $\mathbb{R}^d$-valued, and has $N$ finite support,

$$
X \in \mathcal X = \{x_1,x_2,\ldots,x_N\}. 
$$

The controller chooses $u_t$ from a compact set $\mathcal{U}\subset\mathbb{R}^n$:

$$
u_t=(u_t^1,u_t^2,\ldots,u_t^n) \in \mathcal{U}.
$$

The physical observation is
$$
\boxed{ dY_t = h(u_t,Y_t, X)dt + dW_t, \qquad Y_t, W_t \in \mathbb{R}^m. }
$$

The posterior probabilities are

$$
P_t^i = \mathbb{P}(X=x_i \mid \mathcal{F}_t^Y), \qquad i=1, \ldots, N,
$$

and

$$
P_t = (P_t^1, \ldots, P_t^N) \in \Delta^{N-1}.
$$

The concatenated state is

$$
S_t = (Y_t, P_t) \in \mathbb{R}^m \times \Delta^{N-1}.
$$

In [ ]:
# ==================================================================
# GLOBAL MODEL DIMENSIONS
# ==================================================================
T = 1.0

# Mathematical dimensions.
d = 2      # hidden-state dimension: X in R^d
m = 2      # observation / innovation dimension
n = 2      # control dimension
N = 4      # number of hidden support points
N_T = 20   # number of time steps

DT = T / N_T

STATE_DIM = m + N
TANGENT_DIM = N - 1

assert N >= 2, "The geometric simplex implementation assumes N >= 2."
assert m >= 1
assert n >= 1
assert d >= 1
assert N_T >= 1

In [ ]:
# ==================================================================
# GLOBAL INITIAL DATA
# ==================================================================


# ------------------------------------------------------------------
# Todo: replace X_SUPPORT by the actual finite support. 
# The all-zero tensor is only a structural placeholder.
X_SUPPORT = torch.zeros(
    N,
    d,
    dtype=DTYPE,
    device=DEVICE,
)

# ------------------------------------------------------------------
# Todo: replace P0 by the actual prior
# A uniform prior is only a valid default and automatically adapts to N.

P0 = torch.full(
    (N,),
    1.0 / N,
    dtype=DTYPE,
    device=DEVICE,
)

# ------------------------------------------------------------------
# Todo: replace Y0 by the actual initial condition of the observation process.
# The all-zero tensor is only a placeholder.
Y0 = torch.zeros(
    m,
    dtype=DTYPE,
    device=DEVICE,
)

S0 = torch.cat(
    [Y0, P0],
    dim=0,
)

In [ ]:
# ==================================================================
# GLOBAL CONTROL SET
# ==================================================================

# The default implementation treats U as a hyper-rectangle.
# For a non-box compact set, replace `project_control` below in Section 8.5.
CONTROL_LOW = -torch.ones(
    n,
    dtype=DTYPE,
    device=DEVICE,
)

CONTROL_HIGH = torch.ones(
    n,
    dtype=DTYPE,
    device=DEVICE,
)

assert torch.all(
    CONTROL_HIGH > CONTROL_LOW
)


# ==================================================================
# GLOBAL AUXILIARY CONTROL
# ==================================================================

# A deterministic auxiliary path of shape [N_T, n].
#
# Default: midpoint of the control box at every time.
#
# IMPORTANT:
# This is only the AUXILIARY process used by the classical 2BSDE.
# It is not the Hamiltonian minimiser.
CONTROL_MIDPOINT = 0.5 * (
    CONTROL_LOW + CONTROL_HIGH
)

AUXILIARY_CONTROL_PATH = (
    CONTROL_MIDPOINT[None, :] # shape [1, n]
    .repeat(N_T, 1) # shape [N_T, n]
    .clone()
)


# ==================================================================
# GLOBAL NEURAL-NETWORK HYPERPARAMETERS
# ==================================================================

NETWORK_WIDTH = 64
NETWORK_DEPTH = 3

V0_INITIAL_GUESS = 0.0
GAMMA0_INITIAL_SCALE = 0.0


# ==================================================================
# GLOBAL HAMILTONIAN-OPTIMISATION HYPERPARAMETERS
# ==================================================================

# Multistart projected Adam is used for general n-dimensional controls.
#
# Increase H_OPT_STARTS and H_OPT_STEPS for a more accurate inner
# minimisation.  Decrease them for faster exploratory training.
H_OPT_STARTS = 12
H_OPT_STEPS = 30
H_OPT_LR = 5e-2
H_OPT_GRAD_CLIP = 20.0
H_OPT_SEED = 271828


# ==================================================================
# GLOBAL POSTERIOR-STABILISATION SETTINGS
# ==================================================================

POSTERIOR_STABILISER = "clamp_normalise"
POSTERIOR_EPS = 1e-8


# ==================================================================
# GLOBAL OUTER 2BSDE TRAINING HYPERPARAMETERS
# ==================================================================

RUN_TRAINING = False
RUN_EVALUATION = False

BSDE_EPOCHS = 1000
BSDE_BATCH_SIZE = 128
BSDE_LEARNING_RATE = 2e-3
BSDE_GRAD_CLIP = 10.0

VALUE_LOSS_WEIGHT = 1.0
GRADIENT_LOSS_WEIGHT = 1.0

TEST_PATHS = 2000


def validate_global_configuration():
    """Basic shape and probability checks that do not call h, k, or g."""

    assert X_SUPPORT.shape == (N, d)
    assert P0.shape == (N,)
    assert Y0.shape == (m,)
    assert S0.shape == (STATE_DIM,)
    assert AUXILIARY_CONTROL_PATH.shape == (N_T, n)
    assert CONTROL_LOW.shape == (n,)
    assert CONTROL_HIGH.shape == (n,)

    if not torch.all(P0 >= 0):
        raise ValueError("P0 must have non-negative entries.")

    if not torch.allclose(
        P0.sum(),
        torch.tensor(
            1.0,
            dtype=DTYPE,
            device=DEVICE,
        ),
        atol=1e-12,
        rtol=1e-12,
    ):
        raise ValueError("P0 must sum to one.")

    if torch.unique(
        X_SUPPORT,
        dim=0,
    ).shape[0] < N:
        print(
            "WARNING: X_SUPPORT still contains repeated support points. "
            "This is expected for the placeholder; replace it before training."
        )

# validate_global_configuration()

# print()
# print("T           =", T)
# print("N_T         =", N_T)
# print("DT          =", DT)
# print("d,m,n,N     =", (d, m, n, N))
# print("STATE_DIM   =", STATE_DIM)
# print("TANGENT_DIM =", TANGENT_DIM)


# 2. The sensor $h: \mathcal{U} \times \mathbb{R}^m \times \mathcal{X} \to \mathbb{R}^m$
We assume throughout that for all $x \in \mathcal{X}$,
$$
    h(t, u, y, x) = \sum_{i=0}^{\infty} K(t,u) x^i,

$$
where $C_i: [0,T] \times \mathcal{U} \times \mathbb{R}^m \to \mathbb{R}^m$.

## 2.1 Model-specific sensor interface

The numerical solver only needs the values

$$
h^{(i)}(u, y) = h(u, y, x_i), \qquad i = 1, \ldots, N.
$$

The user-facing function below is therefore vectorised over the finite support.
Its final two output dimensions must be

$$ 
N \times m.
$$

All preceding dimensions are arbitrary batch/candidate dimensions.

For example, if `u` and `y` have shapes `[batch, K, n]` and `[batch, K, m]`,
then `sensor_h(...)` must return `[batch, K, N, m]`.


In [3]:
# ==================================================================
# USER MODEL BLOCK: SENSOR h
# ==================================================================

def sensor_h(
    u: torch.Tensor,
    y: torch.Tensor,
    x_support: torch.Tensor = X_SUPPORT,
) -> torch.Tensor:
    """
    Evaluate h(u, y, x_i) for every hidden support point x_i.

    Parameters
    ----------
    u:
        Control tensor with shape
            (..., n).

    y:
        Observation-state tensor with shape
            (..., m).

    x_support:
        Hidden support with shape
            (N, d).

    Returns
    -------
    h_values:
        Tensor with shape
            (..., N, m).

    Notes
    -----
    * The leading dimensions of `u` and `y` must agree.
    """

    raise NotImplementedError(
        "Implement sensor_h(t, u, y, x_support) for the chosen application."
    )


# 3. Kushner-Stratonovich posterior dynamics and the state process

Define $h^{(i)}: \mathcal{U} \times \mathbb{R}^m \times \mathcal{X} \to \mathbb{R}^m$ as

$$
h^{(i)}(u,y) = h(u,y,x_i) \in\mathbb R^m, \qquad i=1,\ldots,N,
$$

and $\bar{h}: \mathbb{R}^m \times \mathbb{R}^N \times \mathcal{U} \to \mathbb{R}^m$ as

$$
\bar h(y,p,u) = \sum_{i=1}^N p_i h^{(i)}(u,y) \in \mathbb{R}^m.
$$

The posterior dynamics are given by the Kushner-Stratonovich equation (in this case an $N$-dimensional SDE), 
$$
dP_t^i = P_t^i \left( h^{(i)}(u_t, Y_t) - \bar{h}(Y_t, P_t, u_t) \right)^\top dI_t, \qquad i = 1, \ldots, N. 
$$

The $I$ is an $\mathbb{R}^m$-valued, $\mathcal{Y}$-Brownian motion (also known as the innovation process), which is related to $Y$ by 

$$
dY_t = \bar{h}(Y_t, P_t, u_t) dt + dI_t. 
$$

Define $B: \mathbb{R}^m \times \mathbb{R}^N \times \mathcal{U} \to \mathbb{R}^{N \times m}$, row by row, by

$$
\boxed{ B_i(y, p ,u) = p_i \left( h^{(i)}(u,y) - \bar{h}(y,p,u) \right)^\top. }
$$

This can be regarded as the **posterior diffusion matrix**.

We cancatenate the observation and posterior into a single state process $S_t=(Y_t,P_t) \in \mathbb{R}^{m+N}$, which satisfies the SDE

$$
\boxed{ dS_t = b(S_t,u_t) dt + \Sigma(S_t, u_t) dI_t }
$$

with

$$
\boxed{ b(s,u) = \begin{pmatrix} \bar{h}(s_{[1:m]}, s_{[m+1:m+N]}, u) \\ 0_N \end{pmatrix} \in \mathbb{R}^{m+N}, }
$$

and

$$
\boxed{ \Sigma(s,u) = \begin{pmatrix} Id_m \\ B(s_{[1:m]}, s_{[m+1:m+N]}, u) \end{pmatrix} \in \mathbb{R}^{(m+N) \times m}, }
$$
where $Id_m$ is the $m \times m$ identity matrix.

## 3.1 Geometric parameterisation of posterior derivatives

Recall that 

$$
z_p^\top \mathbf{1}_N = 0, \quad \Gamma_{pp} \mathbf{1}_N=0, \quad \Gamma_{yp}\mathbf{1}_N=0,
$$

and

$$
\Gamma_{yy} = \Gamma_{yy}^\top, \quad \Gamma_{yp} = \Gamma_{py}^\top, \quad \Gamma_{pp} = \Gamma_{pp}^\top.
$$

For the zero-column-sum constraint, rather than adding these conditions as soft penalties, we use an orthonormal basis

$$
Q\in\mathbb R^{N\times(N-1)}
$$

of the simplex tangent space 
$$
T\Delta^{N-1} = \{p\in\mathbb R^N: p^\top\mathbf 1_N=0\},
$$

so that

$$
Q^\top Q = I_{N-1}, \qquad Q^\top \mathbf{1}_N = 0.
$$

Then

$$
z_p = Q \widetilde{z}_p, \quad \Gamma_{yp} = C Q^\top, \quad \Gamma_{pp} = Q H Q^\top, \qquad H = H^\top, 
$$

where $\widetilde{z}_p \in \mathbb{R}^{N-1}$, $C \in \mathbb{R}^{m \times (N-1)}$, and $H \in \mathbb{R}^{(N-1) \times (N-1)}$ are unconstrained. We shall call them **reduced coordinate vectors/matrices**. The neural network will output these reduced coordinates, and the full $z_p$, $\Gamma_{yp}$, and $\Gamma_{pp}$ are reconstructed from them. 

This enforces all the geometric constraints exactly. Since

$$
dZ_t = A_t dt + \Gamma_t \Sigma_t dI_t, 
$$

which must preserve $Z_p^\top \mathbf{1}_N = 0$, its drift must satisfy

$$
A_p^\top \mathbf{1}_N=0.
$$

The implementation therefore parameterises $A_p$ in the same tangent basis.

Notice also that, $Q Q^\top \in \mathbb{R}^{N \times N}$ is a projection matrix onto the tangent space of the simplex, 
$$
(Q Q^{\top} x)^{\top} \mathbf{1}_N = x^{\top} Q Q^{\top} \mathbf{1}_N = (x^{\top} Q) (Q^{\top} \mathbf{1}_N) = 0, \quad \forall x \in \mathbb{R}^N.
$$


In [4]:
# ==================================================================
# SIMPLEX TANGENT GEOMETRY
# ==================================================================

from curses import raw

from matplotlib.pylab import indices


def make_simplex_tangent_basis(
    num_states: int,
) -> torch.Tensor:
    """
    Construct an orthonormal basis Q for

        {v in R^N : 1^T v = 0}.

    Q has shape [N, N-1].
    """
    # Construct N x N identity matrix. 
    eye = torch.eye(
        num_states,
        dtype=DTYPE,
        device=DEVICE,
    )

    # Columns are e_i - e_N, i=1,...,N-1.
    spanning = (
        eye[:, : num_states - 1]
        - 
        eye[:, -1:].expand(-1, num_states - 1)
    )

    # Use QR decomposition to orthonormalise the spanning set.
    Q, _ = torch.linalg.qr(
        spanning,
        mode="reduced",
    )

    return Q


TANGENT_BASIS = make_simplex_tangent_basis(N)

ONES_N = torch.ones(
    N,
    dtype=DTYPE,
    device=DEVICE,
)

TANGENT_PROJECTOR = (
    TANGENT_BASIS
    @ TANGENT_BASIS.T
)


def tangent_from_coordinates(
    coordinates: torch.Tensor,
) -> torch.Tensor:
    """
    Map intrinsic coordinates (..., N-1) to an ambient tangent
    vector (..., N).

    Mathematically, this is just the linear map tilde{z} in R^{N-1} -> Q tilde{z} in R^N.
    """
    return torch.einsum(
        "...a, ia -> ...i",
        coordinates, #  shape (..., N-1)
        TANGENT_BASIS, #  shape (N, N-1)
    )


def tangent_coordinates(
    ambient_vector: torch.Tensor,
) -> torch.Tensor:
    """
    Map an ambient vector (..., N) to intrinsic tangent coordinates.

    Any normal component parallel to 1_N is automatically discarded.

    Mathematically, this is the linear map z in R^N -> Q^T z in R^{N-1}.
    """
    return torch.einsum(
        "...i, ia -> ...a",
        ambient_vector, #  shape (..., N)
        TANGENT_BASIS, #  shape (N, N-1)
    )


def project_p_vector_to_tangent(
    v: torch.Tensor,
) -> torch.Tensor:
    """
    Orthogonal projection of (..., N) onto T Delta^{N-1}.
    """
    return torch.einsum(
        "ij, ...j -> ...i",
        TANGENT_PROJECTOR, #  shape (N, N)
        v, #  shape (..., N)
    )

def geometric_vector_from_raw(
    raw: torch.Tensor,
) -> torch.Tensor:
    """
    Convert intrinsic vector parameters

        (..., m + N - 1)

    into an ambient vector

        (..., m + N)

    whose posterior part has zero sum.

    Used for Z and A.
    """
    y_part = raw[..., :m]
    p_coordinates = raw[..., m:]

    p_part = tangent_from_coordinates(
        p_coordinates
    )

    return torch.cat(
        [y_part, p_part],
        dim=-1,
    )


def raw_vector_from_ambient(
    ambient: torch.Tensor,
) -> torch.Tensor:
    """
    Convert an ambient vector to the intrinsic representation.
    """
    y_part = ambient[..., :m]
    p_part = ambient[..., m:]

    return torch.cat(
        [
            y_part,
            tangent_coordinates(p_part),
        ],
        dim=-1,
    )


def upper_triangular_indices(
    dim: int,
):
    """List independent upper-triangular indices."""
    return [
        (i, j)
        for i in range(dim)
        for j in range(i, dim)
    ]

REDUCED_VECTOR_DIM = (
    m + TANGENT_DIM
)

YY_TRIANGULAR_INDICES = (
    upper_triangular_indices(m)
)

PP_TANGENT_TRIANGULAR_INDICES = (
    upper_triangular_indices(TANGENT_DIM)
)

YY_SYM_DIM = len(
    YY_TRIANGULAR_INDICES
)

YP_TANGENT_DIM = (
    m * TANGENT_DIM
)

PP_TANGENT_SYM_DIM = len(
    PP_TANGENT_TRIANGULAR_INDICES
)

GAMMA_PARAM_DIM = (
    YY_SYM_DIM
    + YP_TANGENT_DIM
    + PP_TANGENT_SYM_DIM
)


def raw_to_symmetric_matrix(
    raw: torch.Tensor,
    dim: int,
    indices,
) -> torch.Tensor:
    """
    Convert upper-triangular coordinates into a symmetric matrix.
    """
    expected = dim * (dim + 1) // 2

    assert raw.shape[-1] == expected
    assert len(indices) == expected
    
    matrix = torch.zeros(
        *raw.shape[:-1],
        dim,
        dim,
        dtype=raw.dtype,
        device=raw.device,
    )

    for k, (i, j) in enumerate(indices):
        matrix[..., i, j] = raw[..., k]
        matrix[..., j, i] = raw[..., k]

    return matrix


def symmetric_matrix_to_raw(
    matrix: torch.Tensor,
    indices,
) -> torch.Tensor:
    """
    Extract upper-triangular independent entries.
    """
    return torch.stack(
        [
            matrix[..., i, j]
            for i, j in indices
        ],
        dim=-1,
    )


def geometric_gamma_from_raw(
    raw: torch.Tensor,
) -> torch.Tensor:
    """
    Construct an ambient Hessian Gamma with all geometric
    constraints enforced exactly.

    Parameter blocks
    ----------------
    1. Gamma_yy:
         arbitrary symmetric m x m matrix.

    2. Gamma_yp:
         C Q^T, with C in R^{m x (N-1)}.
         Hence Gamma_yp 1_N = 0.

    3. Gamma_pp:
         Q H Q^T, with H symmetric in R^{(N-1)x(N-1)}.
         Hence, Gamma_pp is symmetric and Gamma_pp 1_N = 0.
    """
    if raw.shape[-1] != GAMMA_PARAM_DIM:
        raise ValueError(
            f"Expected final raw Gamma dimension {GAMMA_PARAM_DIM}, "
            f"got {raw.shape[-1]}."
        )

    offset = 0

    raw_yy = raw[
        ...,
        offset : offset + YY_SYM_DIM,
    ]
    offset += YY_SYM_DIM

    raw_yp = raw[
        ...,
        offset : offset + YP_TANGENT_DIM,
    ]
    offset += YP_TANGENT_DIM

    raw_pp = raw[
        ...,
        offset : offset + PP_TANGENT_SYM_DIM,
    ]

    gamma_yy = raw_to_symmetric_matrix(
        raw_yy,
        m,
        YY_TRIANGULAR_INDICES,
    )

    C = raw_yp.reshape(
        *raw.shape[:-1],
        m,
        TANGENT_DIM,
    ) # shape (..., m, N-1)

    gamma_yp = torch.einsum(
        "...ma, ia -> ...mi",
        C, # shape (..., m, N-1)
        TANGENT_BASIS, # shape (N, N-1)
    )

    H_tangent = raw_to_symmetric_matrix(
        raw_pp,
        TANGENT_DIM,
        PP_TANGENT_TRIANGULAR_INDICES,
    ) # shape (..., N-1, N-1)

    gamma_pp = torch.einsum(
        "ia, ...ab, jb -> ...ij",
        TANGENT_BASIS, # shape (N, N-1)
        H_tangent, # shape (..., N-1, N-1)
        TANGENT_BASIS, # shape (N, N-1)
    )

    gamma = torch.zeros(
        *raw.shape[:-1],
        STATE_DIM,
        STATE_DIM,
        dtype=raw.dtype,
        device=raw.device,
    )

    gamma[..., :m, :m] = gamma_yy
    gamma[..., :m, m:] = gamma_yp
    gamma[..., m:, :m] = gamma_yp.transpose(-1, -2)
    gamma[..., m:, m:] = gamma_pp

    return gamma


def project_gradient_geometry(
    z: torch.Tensor,
) -> torch.Tensor:
    """
    Canonical ambient representative of a gradient on
    R^m x Delta^{N-1}.

    The observation derivative is unchanged; the posterior
    derivative is projected to zero-sum tangent form.
    """
    return torch.cat(
        [
            z[..., :m],
            project_p_vector_to_tangent(
                z[..., m:]
            ),
        ],
        dim=-1,
    )


def project_hessian_geometry(
    gamma: torch.Tensor,
) -> torch.Tensor:
    """
    Orthogonally project an arbitrary ambient Hessian to the
    geometric Hessian class.

    This is useful for diagnostics, 
    or for projecting externally supplied Hessian initialisations.
    """
    gamma = 0.5 * (
        gamma
        + gamma.transpose(-1, -2)
    )

    yy = gamma[..., :m, :m]

    yp = torch.einsum(
        "...mi, ij -> ...mj",
        gamma[..., :m, m:], # shape (..., m, N)
        TANGENT_PROJECTOR, # shape (N, N)
    )

    pp = torch.einsum(
        "ij, ...jk, kl -> ...il",
        TANGENT_PROJECTOR, # shape (N, N)
        gamma[..., m:, m:], # shape (..., N, N)
        TANGENT_PROJECTOR, # shape (N, N)
    )

    result = torch.zeros_like(gamma)

    result[..., :m, :m] = yy
    result[..., :m, m:] = yp
    result[..., m:, :m] = yp.transpose(-1, -2)
    result[..., m:, m:] = pp

    return result


def geometry_residuals(
    z: torch.Tensor,
    gamma: torch.Tensor,
    A: Optional[torch.Tensor] = None,
) -> Dict[str, float]:
    """
    Return maximum absolute violations of the imposed geometry.
    """
    residuals = {
        "z_p_sum":
            float(
                z[..., m:]
                .sum(dim=-1)
                .abs()
                .max()
                .detach()
                .cpu()
            ),
        "gamma_symmetry":
            float(
                (
                    gamma
                    - gamma.transpose(-1, -2)
                )
                .abs()
                .max()
                .detach()
                .cpu()
            ),
        "gamma_yp_row_sum":
            float(
                gamma[..., :m, m:]
                .sum(dim=-1)
                .abs()
                .max()
                .detach()
                .cpu()
            ),
        "gamma_pp_row_sum":
            float(
                gamma[..., m:, m:]
                .sum(dim=-1)
                .abs()
                .max()
                .detach()
                .cpu()
            ),
        "gamma_pp_col_sum":
            float(
                gamma[..., m:, m:]
                .sum(dim=-2)
                .abs()
                .max()
                .detach()
                .cpu()
            ),
    }

    if A is not None:
        residuals["A_p_sum"] = float(
            A[..., m:]
            .sum(dim=-1)
            .abs()
            .max()
            .detach()
            .cpu()
        )

    return residuals


# Numerical self-check of Q.
print(
    "max |Q^T Q - I| =",
    (
        TANGENT_BASIS.T
        @ TANGENT_BASIS
        - torch.eye(
            TANGENT_DIM,
            dtype=DTYPE,
            device=DEVICE,
        )
    )
    .abs()
    .max()
    .item(),
)

print(
    "max |Q^T 1|     =",
    (
        TANGENT_BASIS.T
        @ ONES_N
    )
    .abs()
    .max()
    .item(),
)

print(
    "reduced vector dim =",
    REDUCED_VECTOR_DIM,
)

print(
    "geometric Gamma parameter dim =",
    GAMMA_PARAM_DIM,
)

print(
    "full symmetric Gamma dim     =",
    STATE_DIM, "*", (STATE_DIM + 1), "/", 2 , " = ", STATE_DIM * (STATE_DIM + 1) // 2
)


max |Q^T Q - I| = 4.440892098500626e-16
max |Q^T 1|     = 2.220446049250313e-16
reduced vector dim = 5
geometric Gamma parameter dim = 15
full symmetric Gamma dim     = 6 * 7 / 2  =  21


In [5]:
# ==================================================================
# KS / SEPARATED-STATE DYNAMICS
# ==================================================================

def split_state(
    state: torch.Tensor,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Split S=(Y,P).

    Input
    -----
    state:
        (..., m+N)

    Returns
    -------
    y:
        (..., m)

    p:
        (..., N)
    """
    return (
        state[..., :m],
        state[..., m:],
    )


def posterior_mean_sensor_and_B(
    y: torch.Tensor,
    p: torch.Tensor,
    u: torch.Tensor,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Compute

        h_bar(y,p,u)
        and
        B(y,p,u).

    Shapes
    ------
    y : (..., m)
    p : (..., N)
    u : (..., n)

    returns
    -------
    h_bar : (..., m)
    B     : (..., N, m)

    The left multiplication by TANGENT_PROJECTOR is numerically
    redundant in exact arithmetic, but guarantees

        1_N^T B = 0

    to floating-point precision even if h is complicated.
    """
    h_values = sensor_h(
        u,
        y,
        X_SUPPORT,
    )

    expected_suffix = (N, m)

    if h_values.shape[-2:] != expected_suffix:
        raise ValueError(
            "sensor_h must return shape (..., N, m). "
            f"Expected final dimensions {expected_suffix}, "
            f"got {h_values.shape[-2:]}."
        )

    h_bar = (
        p[..., :, None] # shape (..., N, 1)
        * h_values # shape (..., N, m)
    ).sum(dim=-2) # shape (..., m)

    B_raw = (
        p[..., :, None] # shape (..., N, 1)
        * (
            h_values # shape (..., N, m)
            - 
            h_bar[..., None, :] # shape (..., 1, m)
        )
    ) # shape (..., N, m)

    B = torch.einsum(
        "ij, ...ja -> ...ia",
        TANGENT_PROJECTOR, # shape (N, N) 
        B_raw, # shape (..., N, m)
    )

    return h_bar, B


def state_dynamics(
    state: torch.Tensor,
    u: torch.Tensor,
) -> Tuple[
    torch.Tensor,
    torch.Tensor,
    torch.Tensor,
]:
    """
    Return b(S,u), Sigma(S,u), and B(S,u).

    For

        S=(Y,P),

    we implement

        b =
        [ h_bar
          0_N   ],

    and

        Sigma =
        [ I_m
          B   ].

    Shapes
    ------
    state : (..., m+N)
    u     : (..., n)

    returns
    -------
    drift : (..., m+N)
    Sigma : (..., m+N, m)
    B     : (..., N, m)
    """
    y, p = split_state(state)

    h_bar, B = (
        posterior_mean_sensor_and_B(
            y,
            p,
            u,
        )
    )

    drift = torch.cat(
        [
            h_bar,
            torch.zeros_like(p),
        ],
        dim=-1,
    )

    leading_shape = state.shape[:-1]

    eye = torch.eye(
        m,
        dtype=state.dtype,
        device=state.device,
    ).reshape(
        *([1] * len(leading_shape)),
        m,
        m,
    ).expand(
        *leading_shape,
        m,
        m,
    ) # shape (..., m, m)

    Sigma = torch.cat(
        [eye, B],
        dim=-2,
    )

    return drift, Sigma, B


def euclidean_project_simplex(
    p: torch.Tensor,
) -> torch.Tensor:
    """
    Euclidean projection onto

        {p_i >= 0, sum_i p_i = 1}.

    Vectorised over all leading dimensions.

    NOTICE: This is not the geometric projection used in the 2BSDE, 
                which is onto the tangent space of the simplex. 

            It projects the point onto the simplex only.
            
            It is only used for stabilising the posterior
              when POSTERIOR_STABILISER="euclidean".
    """
    sorted_p, _ = torch.sort(
        p,
        dim=-1,
        descending=True,
    )

    cssv = (
        torch.cumsum(
            sorted_p,
            dim=-1,
        )
        - 1.0
    )

    indices = torch.arange(
        1,
        N + 1,
        dtype=p.dtype,
        device=p.device,
    )

    condition = (
        sorted_p
        - cssv / indices
        > 0
    )

    rho = (
        condition.sum(dim=-1)
        - 1
    ).clamp(min=0)

    theta_all = (
        cssv / indices
    )

    theta = torch.gather(
        theta_all,
        dim=-1,
        index=rho.unsqueeze(-1),
    ).squeeze(-1)

    return torch.clamp(
        p - theta.unsqueeze(-1),
        min=0.0,
    )


def stabilise_posterior(
    p: torch.Tensor,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Correct small Euler-discretisation violations of the simplex.

    Available modes
    ---------------
    "none":
        No correction.

    "clamp_normalise":
        Clamp each p_i below by POSTERIOR_EPS and renormalise.

    "euclidean":
        Euclidean simplex projection.

    Returns
    -------
    corrected_p:
        Same shape as p.

    correction_rate:
        Fraction of paths that required a visible correction.
    """
    visibly_invalid = (
        (p < -1e-10).any(dim=-1)
        | (
            p.sum(dim=-1)
            - 1.0
        ).abs().gt(1e-8)
    )

    if POSTERIOR_STABILISER == "none":
        corrected = p

    elif POSTERIOR_STABILISER == "clamp_normalise":
        corrected = torch.clamp(
            p,
            min=POSTERIOR_EPS,
        )

        corrected = (
            corrected
            / corrected.sum(
                dim=-1,
                keepdim=True,
            )
        )

    elif POSTERIOR_STABILISER == "euclidean":
        corrected = (
            euclidean_project_simplex(p)
        )

    else:
        raise ValueError(
            "Unknown POSTERIOR_STABILISER: "
            f"{POSTERIOR_STABILISER}"
        )

    correction_rate = (
        visibly_invalid
        .to(p.dtype)
        .mean()
    )

    return corrected, correction_rate


def stabilise_state(
    state: torch.Tensor,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Apply posterior stabilisation while leaving Y unchanged."""
    y, p = split_state(state)

    p, correction_rate = (
        stabilise_posterior(p)
    )

    return (
        torch.cat(
            [y, p],
            dim=-1,
        ),
        correction_rate,
    )


# 4. Cost functional

We minimise

$$
\boxed{ J(u) = \mathbb{E} \left[ \int_0^T k(S_t, u_t) dt + g(S_T) \right]. }
$$

## 4.1 Model-specific cost interface

Only two scalar functions are required:

$$
k(s,u)
$$

and

$$
g(s).
$$

The terminal gradient target is computed automatically by PyTorch autograd and
then projected onto the canonical simplex-tangent representative.

Consequently, the user normally does **not** need to hand-code $D_sg$.


In [6]:
# ==================================================================
# USER MODEL BLOCK 2: RUNNING COST k
# ==================================================================

def running_cost_k(
    state: torch.Tensor,
    u: torch.Tensor,
) -> torch.Tensor:
    """
    Running cost k(t,S,u).

    Required input shapes
    ---------------------
    state:
        (..., STATE_DIM)

    u:
        (..., n)

    Required output shape
    ---------------------
        (...)

    The implementation must support arbitrary leading dimensions
    because the Hamiltonian optimiser evaluates many controls in
    parallel.

    Example
    -------
    # y, p = split_state(state)
    # return (
    #     q_y * y.square().sum(dim=-1)
    #     + c_u * u.square().sum(dim=-1)
    #     + posterior_penalty(p)
    # )
    """

    raise NotImplementedError(
        "Implement running_cost_k(t, state, u) for the chosen application."
    )


# ==================================================================
# USER MODEL BLOCK 3: TERMINAL COST g
# ==================================================================

def terminal_cost_g(
    state: torch.Tensor,
) -> torch.Tensor:
    """
    Terminal cost g(S_T).

    Input
    -----
    state:
        (..., STATE_DIM)

    Output
    ------
        (...)

    The function should be differentiable in `state`; the code below
    obtains D_s g automatically with autograd.
    """

    raise NotImplementedError(
        "Implement terminal_cost_g(state) for the chosen application."
    )


def terminal_gradient_g(
    state: torch.Tensor,
) -> torch.Tensor:
    """
    Compute the canonical tangent representative of D_s g.

    If g is originally written as a function on the simplex, an
    ambient extension can have a posterior-gradient component parallel
    to 1_N.  That component has no intrinsic meaning on the simplex.
    We therefore project the posterior derivative onto

        {v : 1_N^T v = 0}.
    """
    with torch.enable_grad():
        state_for_grad = (
            state.detach()
            .clone()
            .requires_grad_(True)
        )

        terminal_value = (
            terminal_cost_g(
                state_for_grad
            )
        )

        gradient = torch.autograd.grad(
            terminal_value.sum(),
            state_for_grad,
            create_graph=False,
            retain_graph=False,
        )[0]

    return (
        project_gradient_geometry(
            gradient
        )
        .detach()
    )


# 5. HJB equation and Hamiltonian

Assume the initial condition is $S_0 = s$. The value function is


$$
\boxed{ V(t,s) = \inf_u \mathbb{E} \left[ \int_t^T k(S_r, u_r) dr + g(S_T) \right]. }
$$

Let

$$
z \in \mathbb{R}^{(m+N)}, \qquad \Gamma \in \mathbb{R}^{(m+N) \times (m+N)}. 
$$

For one candidate control $u$, define the Hamiltonian by 

$$
\boxed{ H(s, z, \Gamma, u) = k(s, u) + b(s, u)^\top z + \frac{1}{2} \operatorname{Tr} \left( \Sigma(s,u)^\top \Gamma \Sigma(s,u) \right). }
$$

Define the minimised Hamiltonian by

$$
\boxed{ \mathcal{H}(s, z, \Gamma) = \min_{u \in \mathcal{U}} H(s, z, \Gamma, u). }
$$

The $z$ and $\Gamma$ can be regarded as the gradient and Hessian of $V$ with respect to $s$, evaluated at $(t,s)$,  

$$
z = D_s V(t, s) \in \mathbb{R}^{(m+N)}, \qquad \Gamma = D_{ss}^2 V(t, s) \in \mathbb{R}^{(m+N) \times (m+N)}. 
$$

Therefore, in practice, in coding, the Hamiltonian $H$ and the minimised Hamiltonian $\mathcal{H}$ will vary with $t$, though they may not be explicitly dependent on $t$.

Then, the Hamilton-Jacobi-Bellman (HJB) equation is 

$$
\boxed{ \partial_t V(t,s) + \mathcal{H} \left( s, D_s V(t,s), D_{ss}^2 V(t,s) \right) = 0, \qquad V(T,s) = g(s). }
$$

Notice that, for fixed $(t,s)$, if we look into $\Gamma$ in blocks, 
$$
\Gamma = D_{ss}^2 V = 
\begin{pmatrix} \Gamma_{yy} & \Gamma_{yp} \\ \Gamma_{py} & \Gamma_{pp} \end{pmatrix} =
\begin{pmatrix} D_{yy}^2 V & D_{yp}^2 V \\ D_{py}^2 V & D_{pp}^2 V \end{pmatrix}, \quad 
\Gamma_{yy} \in \mathbb{R}^{m \times m}, \quad
\Gamma_{yp} \in \mathbb{R}^{m \times N}, \quad
\Gamma_{py} \in \mathbb{R}^{N \times m}, \quad
\Gamma_{pp} \in \mathbb{R}^{N \times N}.
$$

Since $D^2_{ss}V$ is symmetric, we have $\Gamma_{yy}$ symmetric and $\Gamma_{yp} = (\Gamma_{py})^\top$. And the $\Gamma_{pp}$ is also symmetric, with row and column sums equal to zero, since $P$ is moving on a probability simplex, i.e., 
$$
\boxed{ \Gamma_{pp} \mathbf{1}_N = 0, \qquad \mathbf{1}_N \text{ is the N dimensional vector of all ones.} }
$$
A geometric property also holds for $z_p$, the parts associated with the derivative of the value function with respect to $p$, which should be a vector in the tangent space of the probability simplex, i.e., 
$$
\boxed{ z_p^\top \mathbf{1}_N = 0. }
$$

A geometric property also holds for $\Gamma_{yp}$ (equivalently for $\Gamma_{py}^{\top}$). We know $D_p V^{\top} \mathbf{1}_N = 0$. DIfferentiate this with respect to $y_k$, 
$$
\frac{\partial}{\partial y_k} (D_p V^{\top} \mathbf{1}_N) = \sum_{i=1}^N \frac{\partial^2 V}{\partial y_k \partial p_i} = 0, \quad \text{for } k=1,\ldots,m, 
$$
i.e., 
$$
\boxed{ \Gamma_{yp} \mathbf{1}_N = 0, \qquad \text{equivalently } \mathbf{1}_N^\top \Gamma_{py} = 0. }
$$

## 5.1 General multidimensional Hamiltonian minimisation

The implementation uses **multistart projected Adam**:

1. generate quasi-random Sobol initial controls in the bounding box;
2. optimise all starts in parallel;
3. project every iterate back to the admissible control set;
4. keep the best local minimum.

For the default hyper-rectangular $\mathcal U$, projection is coordinate-wise
clipping. For another compact control set, replace only `project_control`.

The inner optimiser uses detached copies of $(s,z,\Gamma)$ to find $u^*$.
Afterwards the Hamiltonian is re-evaluated at the selected, detached $u^*$ using
the original $(s,z,\Gamma)$. This is the appropriate envelope-style computation
for the outer 2BSDE training and avoids differentiating through the entire
inner optimisation algorithm.

The method is still an **approximate global minimiser** for non-convex
Hamiltonians; accuracy is controlled mainly by `H_OPT_STARTS` and
`H_OPT_STEPS`.


In [7]:
# ==================================================================
# HAMILTONIAN
# ==================================================================

def hamiltonian(
    state: torch.Tensor,
    z: torch.Tensor,
    gamma: torch.Tensor,
    u: torch.Tensor,
) -> torch.Tensor:
    """
    Evaluate

        H = k
            + b^T z
            + 1/2 Tr(Sigma^T Gamma Sigma).

    All inputs may carry arbitrary matching leading dimensions.

    Shapes
    ------
    state : (..., STATE_DIM)
    z     : (..., STATE_DIM)
    gamma : (..., STATE_DIM, STATE_DIM)
    u     : (..., n)

    returns
    -------
    H     : (...)
    """
    drift, Sigma, _ = (
        state_dynamics(
            state,
            u,
        )
    )

    running = running_cost_k(
        state,
        u,
    )

    linear = (
        drift * z
    ).sum(dim=-1)

    quadratic = 0.5 * torch.einsum(
        "...ia, ...ij, ...ja -> ...",
        Sigma,
        gamma,
        Sigma,
    )

    return (
        running
        + linear
        + quadratic
    )


# ==================================================================
# GENERAL CONTROL-SET INTERFACE
# ==================================================================

def project_control(
    u: torch.Tensor,
) -> torch.Tensor:
    """
    Project controls to the admissible set U.

    Default:
        hyper-rectangle
        [CONTROL_LOW, CONTROL_HIGH].

    To use another compact set, replace this function.

    Examples of possible replacements:
    * Euclidean ball projection;
    * simplex projection;
    * projection onto linear inequality constraints;
    * a differentiable parameterisation of a manifold-valued control.
    """
    return torch.maximum(
        torch.minimum(
            u,
            CONTROL_HIGH,
        ),
        CONTROL_LOW,
    )


def sobol_control_starts(
    num_starts: int,
) -> torch.Tensor:
    """
    Deterministic quasi-random initial controls.

    Output shape:
        [num_starts, n].

    The first start is always the midpoint of the control box.
    """
    engine = torch.quasirandom.SobolEngine(
        dimension=n,
        scramble=True,
        seed=H_OPT_SEED,
    )

    unit = engine.draw(
        num_starts
    ).to(
        dtype=DTYPE,
        device=DEVICE,
    )

    starts = (
        CONTROL_LOW
        + (
            CONTROL_HIGH
            - CONTROL_LOW
        ) * unit
    )

    starts = project_control(starts)

    starts[0] = CONTROL_MIDPOINT

    return starts


def optimise_control(
    state: torch.Tensor,
    z: torch.Tensor,
    gamma: torch.Tensor,
    *,
    num_starts: Optional[int] = None,
    num_steps: Optional[int] = None,
    learning_rate: Optional[float] = None,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Approximate

        min_{u in U} H(t,S,z,Gamma,u)

    for every element of a batch.

    Parameters
    ----------
    state:
        [batch, STATE_DIM]

    z:
        [batch, STATE_DIM]

    gamma:
        [batch, STATE_DIM, STATE_DIM]

    Returns
    -------
    h_min:
        [batch]

    u_star:
        [batch, n]

    Important implementation detail
    -------------------------------
    The inner optimisation is performed on detached state / derivative
    tensors.  Once the approximate argmin is found, H is evaluated
    again with the original `z` and `gamma`.  Hence the outer 2BSDE
    gradient does not backpropagate through the optimisation trajectory.
    """
    if state.ndim != 2:
        raise ValueError(
            "optimise_control currently expects a batch tensor "
            "[batch, STATE_DIM]."
        )

    K = (
        H_OPT_STARTS
        if num_starts is None
        else num_starts
    )

    steps = (
        H_OPT_STEPS
        if num_steps is None
        else num_steps
    )

    lr = (
        H_OPT_LR
        if learning_rate is None
        else learning_rate
    )

    batch = state.shape[0]

    starts = (
        sobol_control_starts(K)
        [None, :, :]
        .expand(batch, -1, -1)
        .clone()
    )

    # Independent control variables for every batch element and start.
    control_variable = nn.Parameter(
        starts
    )

    inner_optimizer = optim.Adam(
        [control_variable],
        lr=lr,
    )

    # Detached quantities are used only to identify the argmin.
    state_detached = state.detach()
    z_detached = z.detach()
    gamma_detached = gamma.detach()

    state_candidates = (
        state_detached[:, None, :]
        .expand(-1, K, -1)
    )

    z_candidates = (
        z_detached[:, None, :]
        .expand(-1, K, -1)
    )

    gamma_candidates = (
        gamma_detached[:, None, :, :]
        .expand(-1, K, -1, -1)
    )

    with torch.enable_grad():

        for _ in range(steps):

            feasible_control = (
                project_control(
                    control_variable
                )
            )

            H_candidates = hamiltonian(
                state_candidates,
                z_candidates,
                gamma_candidates,
                feasible_control,
            )

            # Every candidate is independent. Averaging only sets the
            # gradient scale for Adam.
            inner_loss = (
                H_candidates.mean()
            )

            inner_optimizer.zero_grad(
                set_to_none=True
            )

            inner_loss.backward()

            torch.nn.utils.clip_grad_norm_(
                [control_variable],
                H_OPT_GRAD_CLIP,
            )

            inner_optimizer.step()

            # Project the optimiser variables themselves after the step.
            with torch.no_grad():
                control_variable.copy_(
                    project_control(
                        control_variable
                    )
                )

    with torch.no_grad():
        final_candidates = (
            project_control(
                control_variable
            )
        )

        final_values = hamiltonian(
            state_candidates,
            z_candidates,
            gamma_candidates,
            final_candidates,
        )

        best_index = (
            final_values.argmin(
                dim=1
            )
        )

        batch_index = torch.arange(
            batch,
            device=state.device,
        )

        u_star = (
            final_candidates[
                batch_index,
                best_index,
            ]
            .detach()
        )

    # Re-evaluate with the original z and gamma.
    # This preserves the outer dependence of H_min on the 2BSDE
    # variables while treating the numerical argmin as fixed.
    h_min = hamiltonian(
        state,
        z,
        gamma,
        u_star,
    )

    return h_min, u_star


# 6. Classical 2BSDE derivation

We set the auxiliary control process $u^{\mathrm{aux}} = (u^{\mathrm{aux}}_t)_{0 \leq t \leq T}$, where $u^{\mathrm{aux}}_t \in \mathcal{U}$ for each $t$, and define the auxiliary state process $S^{\mathrm{aux}}$ by 
$$
d S^{\mathrm{aux}}_t = b(S_t^{\mathrm{aux}}, u^{\rm aux}_t) dt + \Sigma(S_t^{\mathrm{aux}}, u^{\mathrm{aux}}_t) dI_t.
$$

Set 
$$
b^{\mathrm{aux}}_t = b(S_t^{\mathrm{aux}}, u^{\mathrm{aux}}_t), \qquad \Sigma^{\rm aux}_t = \Sigma(S_t^{\mathrm{aux}}, u^{\mathrm{aux}}_t).
$$

Define the scalar value process along the auxiliary state by 

$$
\boxed{ R_t = V(t, S_t^{\mathrm{aux}}). }
$$

Define

$$
\boxed{ Z_t = D_s V(t, S_t^{\mathrm{aux}}), \qquad \Gamma_t = D_{ss}^2 V(t, S_t^{\mathrm{aux}}). }
$$

Applying Itô's formula, we get 

$$
dR_t = \left[ \partial_t V + {b^{\mathrm{aux}}_t}^\top Z_t + \frac{1}{2} \operatorname{Tr} \left( {\Sigma^{\mathrm{aux}}_t}^\top \Gamma_t \Sigma^{\mathrm{aux}}_t \right) \right] dt + Z_t^\top \Sigma^{\mathrm{aux}}_t dI_t.
$$

The HJB gives

$$
\partial_t V = - \mathcal H(S_t^{\mathrm{aux}}, Z_t, \Gamma_t).
$$

Therefore

$$
dR_t = F_t dt + Z_t^\top \Sigma^{\mathrm{aux}}_t dI_t, 
$$

with

$$
\boxed{ F_t(S_t^{\mathrm{aux}}, Z_t, \Gamma_t) = - \mathcal{H}(S_t^{\mathrm{aux}},Z_t,\Gamma_t) + {b^{\mathrm{aux}}_t}^\top Z_t + \frac{1}{2} \operatorname{Tr} \left( {\Sigma^{\mathrm{aux}}_t}^\top \Gamma_t \Sigma^{\mathrm{aux}}_t \right). }
$$

Applying Itô's formula to the gradient process, we get

$$
dZ_t = A_t dt + \Gamma_t\Sigma^{\mathrm{aux}}_t dI_t. 
$$

The vector $A_t \in \mathbb{R}^{m+N}$ is the drift of the gradient process, and has a complicated analytic expression. It is essentially a function $A_t = \mathcal{L} D_s V(t, S_t^{\mathrm{aux}})$. We are not interested in deriving it analytically though, as we will use a neural network to approximate it.

The solution to the second-order backward SDE (2BSDE) 

\begin{align*}
dR_t &= F_t(S_t^{\mathrm{aux}}, Z_t, \Gamma_t) dt + Z_t^\top \Sigma^{\mathrm{aux}}_t dI_t, \quad R_T = g(S_T^{\mathrm{aux}})\\
dZ_t &= A_t dt + \Gamma_t\Sigma^{\mathrm{aux}}_t dI_t, \quad Z_T = \partial_s g(S_T^{\mathrm{aux}}), 
\end{align*}
is a probabilistic representation of the solution to the HJB equation.

We discretise the time interval $[0,T]$ into $N_T$ subintervals of length $\Delta t = T/N_T$. Denote the associated forward discrete-time processes by 

$$
S_{i+1}^{\mathrm{aux}} = S_i^{\mathrm{aux}} + b^{\mathrm{aux}}_i \Delta t + \Sigma^{{\mathrm{aux}}}_{i} \Delta I_i, \quad S_0^{\mathrm{aux}} = s_0,
$$

$$
\mathcal{R}_{i+1} = \mathcal{R}_i + F_i(S_i^{\mathrm{aux}}, \mathcal{Z}_i, \mathbb{G}_i) \Delta t + (\mathcal{Z}_i^\top \Sigma^{{\mathrm{aux}}}_i) \Delta I_i, \quad \mathcal{R}_0 = \mathbb{R}_0,
$$

$$
\mathcal{Z}_{i+1} = \mathcal{Z}_i + \mathbb{A}_i \Delta t + \mathbb{G}_i \Sigma^{\mathrm{aux}}_i \Delta I_i, \quad \mathcal{Z}_0 = \mathbb{Z}_0,
$$
where $S_i$ (similarly for all other subscripted terms) means the $S$ value at time $t = i\Delta t$, $i=0,1,\ldots,N_T-1$.


The minimiser $u_i^*$ is derived inside $\mathcal H$. 

## 6.1 Auxiliary control and geometric neural blocks

The auxiliary control is represented by the global tensor

```python
AUXILIARY_CONTROL_PATH
```

with shape `[N_T, n]`. It may be replaced by any deterministic schedule before
training.

For the neural approximation, each non-initial time block contains two MLPs:

$$
S_i^{\mathrm{aux}}\longmapsto A_i,
$$

and

$$
S_i^{\mathrm{aux}}\longmapsto \Gamma_i.
$$

Their raw outputs are lower-dimensional intrinsic coordinates; the geometric
mapping then reconstructs the ambient $A_i$ and $\Gamma_i$.

For example, with general $m,N$:

- an ambient $Z$ or $A$ has dimension $m+N$, but only
  $m+N-1$ intrinsic coordinates are trained;
- a full symmetric ambient Hessian has
  $(m+N)(m+N+1)/2$ entries;
- the geometric Hessian head outputs only

$$
\frac{m(m+1)}{2} + m(N-1) + \frac{(N-1)N}{2}
$$

independent parameters.


In [8]:
# ==================================================================
# AUXILIARY CONTROL
# ==================================================================

def auxiliary_control(
    step: int,
    batch_size: int,
) -> torch.Tensor:
    """
    Deterministic reference control at one time block.

    Returns
    -------
    u_aux:
        [batch_size, n]
    """
    if not (
        0 <= step < N_T
    ):
        raise IndexError(
            f"step must be in [0,{N_T-1}]"
        )

    value = (
        AUXILIARY_CONTROL_PATH[
            step
        ]
    )

    value = project_control(
        value
    )

    return value.expand(
        batch_size,
        -1,
    )


# ==================================================================
# MLP BUILDING BLOCKS
# ==================================================================

def make_mlp(
    input_dim: int,
    output_dim: int,
    width: int = NETWORK_WIDTH,
    depth: int = NETWORK_DEPTH,
) -> nn.Module:
    """
    Standard fully-connected tanh MLP.

    `depth` counts hidden layers.
    """
    layers = []

    current_dim = input_dim

    for _ in range(depth):
        layers.extend(
            [
                nn.Linear(
                    current_dim,
                    width,
                ),
                nn.Tanh(),
            ]
        )

        current_dim = width

    layers.append(
        nn.Linear(
            current_dim,
            output_dim,
        )
    )

    return nn.Sequential(*layers)


class GeometricParallelBlock(nn.Module):
    """
    One time-layer block:

        S -> A(S)
        S -> Gamma(S)

    Geometry is imposed exactly after the raw neural output.
    """

    def __init__(
        self,
        width: int = NETWORK_WIDTH,
        depth: int = NETWORK_DEPTH,
    ):
        super().__init__()

        self.a_net = make_mlp(
            STATE_DIM,
            REDUCED_VECTOR_DIM,
            width=width,
            depth=depth,
        )

        self.gamma_net = make_mlp(
            STATE_DIM,
            GAMMA_PARAM_DIM,
            width=width,
            depth=depth,
        )

    def forward(
        self,
        state: torch.Tensor,
    ) -> Tuple[
        torch.Tensor,
        torch.Tensor,
    ]:
        raw_A = self.a_net(state)

        raw_gamma = (
            self.gamma_net(state)
        )

        A = geometric_vector_from_raw(
            raw_A
        )

        gamma = (
            geometric_gamma_from_raw(
                raw_gamma
            )
        )

        return A, gamma


# ------------------------------------------------------------------
# Geometry-only smoke test.
#
# This can run before h, k, and g are implemented.
# ------------------------------------------------------------------
geometry_probe_block = (
    GeometricParallelBlock(
        width=16,
        depth=2,
    )
    .to(
        device=DEVICE,
        dtype=DTYPE,
    )
)

geometry_probe_state = torch.randn(
    5,
    STATE_DIM,
    dtype=DTYPE,
    device=DEVICE,
)

with torch.no_grad():
    geometry_probe_A, geometry_probe_gamma = (
        geometry_probe_block(
            geometry_probe_state
        )
    )

    geometry_probe_z = (
        geometric_vector_from_raw(
            torch.randn(
                5,
                REDUCED_VECTOR_DIM,
                dtype=DTYPE,
                device=DEVICE,
            )
        )
    )

print()
print(
    "geometric block residuals:",
    geometry_residuals(
        geometry_probe_z,
        geometry_probe_gamma,
        geometry_probe_A,
    ),
)

del (
    geometry_probe_block,
    geometry_probe_state,
    geometry_probe_A,
    geometry_probe_gamma,
    geometry_probe_z,
)



geometric block residuals: {'z_p_sum': 4.440892098500626e-16, 'gamma_symmetry': 5.551115123125783e-17, 'gamma_yp_row_sum': 5.551115123125783e-17, 'gamma_pp_row_sum': 1.249000902703301e-16, 'gamma_pp_col_sum': 1.1102230246251565e-16, 'A_p_sum': 8.326672684688674e-17}


# 7. Neural network design 
We use $\mathbb{R}_0 \in \mathbb{R}$, $\mathbb{Z}_0 \in \mathbb{R}^{m+N}$, and $\mathbb{A}_i$, $\mathbb{G}_i$, $i=0,1,\ldots,N_T-1$, as neural network parameters. More specifically, $\mathbb{R}_0$ is the guessed intial value of the value function, $\mathbb{Z}_0$ is the guessed initial value of the gradient, and $\mathbb{A}_i$, $\mathbb{G}_i$ are used to approximate at each time $t=i\Delta t$ the map $A(t_i, s)$ and $\Gamma(t_i, s)$. 

In practice, we first choose a deterministic auxiliary control process $u^{aux}$. Initiate the network with the above mentioned parameters. Then, for a given initial state $s_0$ and a sample Brownian increments, 
$$
(\Delta I_0, \Delta I_1, \ldots, \Delta I_{N_T-1}), 
$$

we get 

$$
s_0, \Delta I_0, u^{\mathrm{aux}}_0 \Longrightarrow b^{\mathrm{aux}}_0, \Sigma^{\mathrm{aux}}_0 \Longrightarrow S_1^{\mathrm{aux}},  
$$

$$
s_0, \Delta I_0, \Sigma^{\mathrm{aux}}_0, \mathbb{R}_0, \mathbb{Z}_0, \mathbb{A}_0, \mathbb{G}_0 \Longrightarrow \mathcal{R}_1, \mathcal{Z_1}, 
$$

then 

$$
S_1^{\mathrm{aux}}, \Delta I_1, u^{\mathrm{aux}}_1 \Longrightarrow b^{\mathrm{aux}}_1, \Sigma^{\mathrm{aux}}_1 \Longrightarrow S_2^{\mathrm{aux}},  
$$

$$
S_1^{\mathrm{aux}}, \Delta I_1, \Sigma^{\mathrm{aux}}_1, \mathcal{R}_1, \mathcal{Z}_1, \mathbb{A}_1, \mathbb{G}_1 \Longrightarrow \mathcal{R}_2, \mathcal{Z_2}, 
$$

$$
\cdots, 
$$

$$
S_{N_T-1}^{\mathrm{aux}}, \Delta I_{N_T-1}, u^{\mathrm{aux}}_{N_T-1} \Longrightarrow b^{\mathrm{aux}}_{N_T-1}, \Sigma^{\mathrm{aux}}_{N_T-1} \Longrightarrow S_{N_T}^{\mathrm{aux}},  
$$

$$
S_{N_T-1}^{\mathrm{aux}}, \Delta I_{N_T-1}, \Sigma^{\mathrm{aux}}_{N_T-1}, \mathcal{R}_{N_T-1}, \mathcal{Z}_{N_T-1}, \mathbb{A}_{N_T-1}, \mathbb{G}_{N_T-1} \Longrightarrow \mathcal{R}_{N_T}, \mathcal{Z}_{N_T}.  
$$

We know explcitly that $\mathcal{R}_{N_T} = g(S_{N_T}^{\mathrm{aux}})$ and $\mathcal{Z}_{N_T} = D_s g(S_{N_T}^{\mathrm{aux}})$. Therefore, for $M$ samples,we can define the loss function as
$$
\mathcal{L} = \frac{1}{M} \left( \sum_{m=1}^M \left( \mathcal{R}_{N_T}^{(m)} - g(S_{N_T}^{\mathrm{aux},(m)}) \right)^2 +  \sum_{m=1}^M \left\| \mathcal{Z}_{N_T}^{(m)} - D_s g(S_{N_T}^{\mathrm{aux},(m)}) \right\|^2 \right).
$$

In [9]:
# ==================================================================
# CLASSICAL FIXED-AUXILIARY GEOMETRIC 2BSDE
# ==================================================================

class GeometricClassicalFixedAux2BSDE(nn.Module):
    """
    The forward state always follows the deterministic auxiliary
    control.  The Hamiltonian minimiser u_star is evaluated and may be
    recorded, but DOES NOT generate S_aux in this classical method.
    """

    def __init__(
        self,
        width: int = NETWORK_WIDTH,
        depth: int = NETWORK_DEPTH,
    ):
        super().__init__()

        # Time blocks used after the initial fixed state.
        self.blocks = nn.ModuleList(
            [
                GeometricParallelBlock(
                    width=width,
                    depth=depth,
                )
                for _ in range(
                    max(N_T - 1, 0)
                )
            ]
        )

        self.R0 = nn.Parameter(
            torch.tensor(
                V0_INITIAL_GUESS,
                dtype=DTYPE,
                device=DEVICE,
            )
        )

        # Intrinsic initial gradient parameter.
        self.Z0_raw = nn.Parameter(
            torch.zeros(
                REDUCED_VECTOR_DIM,
                dtype=DTYPE,
                device=DEVICE,
            )
        )

        # Intrinsic initial drift-of-gradient parameter.
        self.A0_raw = nn.Parameter(
            torch.zeros(
                REDUCED_VECTOR_DIM,
                dtype=DTYPE,
                device=DEVICE,
            )
        )

        # Geometric initial Hessian parameter.
        self.Gamma0_raw = nn.Parameter(
            GAMMA0_INITIAL_SCALE
            * torch.randn(
                GAMMA_PARAM_DIM,
                dtype=DTYPE,
                device=DEVICE,
            )
        )

    def Z0(self) -> torch.Tensor:
        """Ambient initial gradient with z_p^T 1_N = 0."""
        return geometric_vector_from_raw(
            self.Z0_raw
        )

    def A0(self) -> torch.Tensor:
        """Ambient initial A with A_p^T 1_N = 0."""
        return geometric_vector_from_raw(
            self.A0_raw
        )

    def Gamma0(self) -> torch.Tensor:
        """Ambient initial Hessian satisfying all Gamma geometry."""
        return geometric_gamma_from_raw(
            self.Gamma0_raw
        )

    def forward(
        self,
        dI: torch.Tensor,
        return_paths: bool = False,
    ):
        """
        Parameters
        ----------
        dI:
            Innovation increments of shape

                [batch, N_T, m].

            Usually sampled as

                sqrt(DT) * Normal(0, I_m).

        return_paths:
            If False, return only terminal quantities needed for
            training.

            If True, also return complete S, R, Z, A, Gamma, u_aux,
            and u_star paths for diagnostics.
        """
        if dI.ndim != 3:
            raise ValueError(
                "dI must have shape [batch, N_T, m]."
            )

        if dI.shape[1:] != (
            N_T,
            m,
        ):
            raise ValueError(
                "dI shape mismatch: expected "
                f"[batch,{N_T},{m}], got {tuple(dI.shape)}."
            )

        batch = dI.shape[0]

        state = (
            S0[None, :]
            .expand(batch, -1)
            .clone()
        )

        R = self.R0.expand(batch)

        Z = (
            self.Z0()[None, :]
            .expand(batch, -1)
        )

        A = (
            self.A0()[None, :]
            .expand(batch, -1)
        )

        Gamma = (
            self.Gamma0()[None, :, :]
            .expand(batch, -1, -1)
        )

        # Complete paths.
        state_path = [state]
        R_path = [R]
        Z_path = [Z]

        A_path = []
        Gamma_path = []
        u_aux_path = []
        u_star_path = []
        correction_rates = []

        auxiliary_running_cost = (
            torch.zeros(
                batch,
                dtype=DTYPE,
                device=DEVICE,
            )
        )

        for step in range(N_T):

            # ------------------------------------------------------
            # 1. Fixed auxiliary control and its state dynamics.
            # ------------------------------------------------------
            u_aux = auxiliary_control(
                step,
                batch,
            )

            (
                drift_aux,
                Sigma_aux,
                _,
            ) = state_dynamics(
                state,
                u_aux,
            )

            # ------------------------------------------------------
            # 2. HJB Hamiltonian minimiser at the CURRENT auxiliary
            #    state and current 2BSDE derivative values.
            #
            #    It is NOT used to advance the classical reference
            #    state.
            # ------------------------------------------------------
            h_min, u_star = optimise_control(
                state,
                Z,
                Gamma,
            )

            # ------------------------------------------------------
            # 3. Classical 2BSDE driver
            #
            # F = -H_min
            #     + b_aux^T Z
            #     + 1/2 Tr(Sigma_aux^T Gamma Sigma_aux).
            # ------------------------------------------------------
            auxiliary_quadratic = torch.einsum(
                "bia, bij, bja -> b",
                Sigma_aux,
                Gamma,
                Sigma_aux,
            )

            driver = (
                -h_min
                + (
                    drift_aux * Z
                ).sum(dim=-1)
                + 0.5
                * auxiliary_quadratic
            )

            Z_sigma = torch.einsum(
                "bi,bia->ba",
                Z,
                Sigma_aux,
            )

            # ------------------------------------------------------
            # 4. Update scalar value process R.
            # ------------------------------------------------------
            R = (
                R
                + driver * DT
                + (
                    Z_sigma
                    * dI[:, step, :]
                ).sum(dim=-1)
            )

            # ------------------------------------------------------
            # 5. Update pathwise gradient Z.
            # ------------------------------------------------------
            Z = (
                Z
                + A * DT
                + torch.einsum(
                    "bij,bja,ba->bi",
                    Gamma,
                    Sigma_aux,
                    dI[:, step, :],
                )
            )

            # The continuous geometry should already preserve the
            # tangent property.  This projection removes only
            # floating-point / time-discretisation normal drift.
            Z = project_gradient_geometry(Z)

            # ------------------------------------------------------
            # 6. Record the cost of the auxiliary reference control.
            #
            # This is useful as a baseline, but is NOT the optimal
            # policy cost.
            # ------------------------------------------------------
            auxiliary_running_cost = (
                auxiliary_running_cost
                + running_cost_k(
                    state,
                    u_aux,
                ) * DT
            )

            # ------------------------------------------------------
            # 7. Advance the REFERENCE state with u_aux.
            # ------------------------------------------------------
            state = (
                state
                + drift_aux * DT
                + torch.einsum(
                    "bia, ba -> bi",
                    Sigma_aux,
                    dI[:, step, :],
                )
            )

            (state, correction_rate, ) = stabilise_state(state)

            # ------------------------------------------------------
            # 8. Save current-layer diagnostics.
            # ------------------------------------------------------
            A_path.append(A)
            Gamma_path.append(Gamma)
            u_aux_path.append(u_aux)
            u_star_path.append(u_star)
            correction_rates.append(
                correction_rate
            )

            state_path.append(state)
            R_path.append(R)
            Z_path.append(Z)

            # ------------------------------------------------------
            # 9. Evaluate the next time-layer networks at the new
            #    auxiliary state.
            # ------------------------------------------------------
            if step < N_T - 1:
                A, Gamma = (
                    self.blocks[step](
                        state
                    )
                )

        auxiliary_reference_cost = (
            auxiliary_running_cost
            + terminal_cost_g(state)
        )

        correction_rate = torch.stack(
            correction_rates
        ).mean()

        if not return_paths:
            return (
                R,
                Z,
                state,
                correction_rate,
            )

        return {
            "R_T": R,
            "Z_T": Z,
            "S_T": state,
            "R_path":
                torch.stack(
                    R_path,
                    dim=1,
                ),
            "Z_path":
                torch.stack(
                    Z_path,
                    dim=1,
                ),
            "S_path":
                torch.stack(
                    state_path,
                    dim=1,
                ),
            "A_path":
                torch.stack(
                    A_path,
                    dim=1,
                ),
            "Gamma_path":
                torch.stack(
                    Gamma_path,
                    dim=1,
                ),
            "u_aux_path":
                torch.stack(
                    u_aux_path,
                    dim=1,
                ),
            "u_star_path":
                torch.stack(
                    u_star_path,
                    dim=1,
                ),
            "auxiliary_reference_cost":
                auxiliary_reference_cost,
            "posterior_correction_rate":
                correction_rate,
        }


# The model can be instantiated before the user fills in h, k, and g.
# A full forward call will require those model-specific functions.
model = (
    GeometricClassicalFixedAux2BSDE(
        width=NETWORK_WIDTH,
        depth=NETWORK_DEPTH,
    )
    .to(
        device=DEVICE,
        dtype=DTYPE,
    )
)

with torch.no_grad():
    initial_geometry = geometry_residuals(
        model.Z0()[None, :],
        model.Gamma0()[None, :, :],
        model.A0()[None, :],
    )

print()
print(
    "initial trainable geometry residuals:",
    initial_geometry,
)



initial trainable geometry residuals: {'z_p_sum': 0.0, 'gamma_symmetry': 0.0, 'gamma_yp_row_sum': 0.0, 'gamma_pp_row_sum': 0.0, 'gamma_pp_col_sum': 0.0, 'A_p_sum': 0.0}


## 8 Training and evaluation

Because `sensor_h`, `running_cost_k`, and `terminal_cost_g` are intentionally left
application-specific, the notebook defaults to

```python
RUN_TRAINING = False
RUN_EVALUATION = False
```

After filling those three functions:

1. set `RUN_TRAINING = True`;
2. run the notebook from the top;
3. optionally set `RUN_EVALUATION = True` for fresh-path diagnostics.

The terminal loss is

$$
\mathcal{L} = w_V \mathbb{E} \left[ \left| \mathcal{R}_{N_T}-g(S_{N_T}^{\mathrm{aux}}) \right|^2 \right] 
+ 
w_Z \mathbb{E} \left[ \left\| \mathcal{Z}_{N_T} - D_s g(S_{N_T}^{\mathrm{aux}}) \right\|^2 \right].
$$

The gradient target uses the tangent projection described above.


In [10]:
# ==================================================================
# TRAINING
# ==================================================================

loss_history = []
value_loss_history = []
gradient_loss_history = []

if RUN_TRAINING:

    outer_optimizer = optim.Adam(
        model.parameters(),
        lr=BSDE_LEARNING_RATE,
    )

    training_start = (
        time.perf_counter()
    )

    for epoch in range(
        1,
        BSDE_EPOCHS + 1,
    ):

        dI_batch = (
            math.sqrt(DT)
            * torch.randn(
                BSDE_BATCH_SIZE,
                N_T,
                m,
                dtype=DTYPE,
                device=DEVICE,
            )
        )

        (
            R_T,
            Z_T,
            S_T,
            correction_rate,
        ) = model(
            dI_batch,
            return_paths=False,
        )

        target_value = (
            terminal_cost_g(S_T)
        )

        target_gradient = (
            terminal_gradient_g(S_T)
        )

        value_loss = (
            R_T
            - target_value
        ).square().mean()

        gradient_loss = (
            Z_T
            - target_gradient
        ).square().mean()

        total_loss = (
            VALUE_LOSS_WEIGHT
            * value_loss
            + GRADIENT_LOSS_WEIGHT
            * gradient_loss
        )

        outer_optimizer.zero_grad(
            set_to_none=True
        )

        total_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            BSDE_GRAD_CLIP,
        )

        outer_optimizer.step()

        loss_history.append(
            total_loss.item()
        )

        value_loss_history.append(
            value_loss.item()
        )

        gradient_loss_history.append(
            gradient_loss.item()
        )

        if (
            epoch == 1
            or epoch % 50 == 0
        ):
            print(
                f"epoch={epoch:5d} | "
                f"loss={total_loss.item():.6e} | "
                f"value={value_loss.item():.6e} | "
                f"gradient={gradient_loss.item():.6e} | "
                f"R0={model.R0.item():.6e} | "
                f"posterior correction="
                f"{correction_rate.item():.3e}"
            )

    training_elapsed = (
        time.perf_counter()
        - training_start
    )

    print()
    print(
        "training time:",
        f"{training_elapsed:.2f} s",
    )

    print(
        "trained R0:",
        model.R0.item(),
    )

else:
    print(
        "Training skipped. "
        "Implement sensor_h, running_cost_k, terminal_cost_g, "
        "then set RUN_TRAINING=True."
    )


Training skipped. Implement sensor_h, running_cost_k, terminal_cost_g, then set RUN_TRAINING=True.


In [11]:
# ==================================================================
# FRESH-PATH EVALUATION / DIAGNOSTICS
# ==================================================================

if RUN_EVALUATION:

    if not RUN_TRAINING:
        print(
            "WARNING: RUN_EVALUATION=True while RUN_TRAINING=False. "
            "The model may be untrained unless weights were loaded manually."
        )

    model.eval()

    dI_test = (
        math.sqrt(DT)
        * torch.randn(
            TEST_PATHS,
            N_T,
            m,
            dtype=DTYPE,
            device=DEVICE,
        )
    )

    result = model(
        dI_test,
        return_paths=True,
    )

    S_path = result["S_path"]
    Z_path = result["Z_path"]
    A_path = result["A_path"]
    Gamma_path = result["Gamma_path"]

    # Geometry at the time layers where Gamma and A are defined.
    geometry = geometry_residuals(
        Z_path[:, :N_T, :],
        Gamma_path,
        A_path,
    )

    print()
    print("fresh-path geometry residuals:")
    for key, value in geometry.items():
        print(
            f"  {key:24s}: {value:.3e}"
        )

    posterior_path = (
        S_path[..., m:]
    )

    print()
    print(
        "max |sum(P)-1|:",
        (
            posterior_path
            .sum(dim=-1)
            - 1.0
        )
        .abs()
        .max()
        .item(),
    )

    print(
        "min posterior entry:",
        posterior_path.min().item(),
    )

    print(
        "mean auxiliary reference cost:",
        result[
            "auxiliary_reference_cost"
        ].mean().item(),
    )

    print(
        "posterior correction rate:",
        result[
            "posterior_correction_rate"
        ].item(),
    )


    # --------------------------------------------------------------
    # Training losses.
    # --------------------------------------------------------------
    if loss_history:

        plt.figure(
            figsize=(7, 4)
        )

        plt.plot(
            loss_history,
            label="total",
        )

        plt.plot(
            value_loss_history,
            label="terminal value",
        )

        plt.plot(
            gradient_loss_history,
            label="terminal gradient",
        )

        plt.yscale("log")
        plt.xlabel("epoch")
        plt.ylabel("loss")
        plt.title(
            "Geometric classical 2BSDE training"
        )
        plt.legend()
        plt.show()


    # --------------------------------------------------------------
    # One sample: every control component over time.
    # Works for arbitrary n.
    # --------------------------------------------------------------
    sample = 0

    t_control = (
        np.arange(N_T)
        * DT
    )

    u_aux_sample = (
        result["u_aux_path"][
            sample
        ]
        .detach()
        .cpu()
        .numpy()
    )

    u_star_sample = (
        result["u_star_path"][
            sample
        ]
        .detach()
        .cpu()
        .numpy()
    )

    plt.figure(
        figsize=(9, 5)
    )

    for j in range(n):

        plt.step(
            t_control,
            u_aux_sample[:, j],
            where="post",
            linestyle="--",
            label=fr"$u^{{aux,{j+1}}}$",
        )

        plt.step(
            t_control,
            u_star_sample[:, j],
            where="post",
            label=fr"$u^{{*,{j+1}}}$",
        )

    plt.xlabel("time")
    plt.ylabel("control")
    plt.title(
        "Auxiliary control vs Hamiltonian minimiser"
    )
    plt.legend(
        ncol=min(2 * n, 4)
    )
    plt.show()


    # Optional 2D control-plane plot when n=2.
    if n == 2:

        plt.figure(
            figsize=(6, 6)
        )

        plt.plot(
            u_aux_sample[:, 0],
            u_aux_sample[:, 1],
            marker="o",
            linestyle="--",
            label=r"$u^{aux}$",
        )

        plt.plot(
            u_star_sample[:, 0],
            u_star_sample[:, 1],
            marker="s",
            label=r"$u^*$",
        )

        plt.xlabel(r"$u^1$")
        plt.ylabel(r"$u^2$")
        plt.title(
            "One sample in the 2D control plane"
        )
        plt.legend()
        plt.gca().set_aspect(
            "equal",
            adjustable="box",
        )
        plt.show()

else:
    print(
        "Evaluation skipped. "
        "Set RUN_EVALUATION=True after the model-specific blocks are implemented."
    )


Evaluation skipped. Set RUN_EVALUATION=True after the model-specific blocks are implemented.


## 9 Conclusion

### What is fully implemented

The notebook now contains dimension-generic code for:

- finite-support KS dynamics for arbitrary $d,m,n,N$;
- the concatenated state $S=(Y,P)$;
- posterior simplex stabilisation;
- intrinsic tangent geometry for $Z_p$ and $A_p$;
- exact geometric parameterisation of $\Gamma_{yp}$ and $\Gamma_{pp}$;
- the Hamiltonian;
- general multidimensional approximate minimisation of the Hamiltonian;
- deterministic auxiliary controls;
- the classical fixed-reference 2BSDE recursion;
- state-dependent neural blocks for $A_i$ and $\Gamma_i$;
- terminal value and terminal-gradient losses;
- automatic terminal differentiation of $g$;
- complete diagnostic paths including `A_path` and `Gamma_path`.

### What is intentionally application-specific

Only the following blocks are left open:

```python
sensor_h(...)
running_cost_k(...)
terminal_cost_g(...)
```

and, when required by a new application,

```python
X_SUPPORT
P0
Y0
AUXILIARY_CONTROL_PATH
project_control(...)
```

### On the Hamiltonian minimiser

For $n=1$ or $n=2$, a dense grid can sometimes be more reliable and cheaper.
For genuinely multidimensional controls, however, the number of grid points
grows exponentially in $n$. The multistart projected optimiser used here avoids
that exponential grid.

For difficult non-convex Hamiltonians, increase:

```python
H_OPT_STARTS
H_OPT_STEPS
```

and compare several seeds. If $\mathcal U$ has special structure or if a
closed-form minimiser is available, replacing `optimise_control` by the
problem-specific solver is preferable.

### On the simplex geometry

The code does **not** learn arbitrary ambient posterior derivatives and then
penalise constraint violations. Instead, it learns coordinates directly in

$$
T\Delta^{N-1}.
$$

Therefore the geometric identities are satisfied up to floating-point
round-off throughout training.
